**Installing the langchain, openai**

In [ ]:
!pip install langchain-ollama

In [ ]:
!pip install --upgrade --quiet langchain-google-genai langchain-openai

**Importing the library**

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

##Retrieve the data


In [ ]:
! curl "https://api.mockaroo.com/api/dde01370?count=1000&key=11149690" > "customers.csv"
! curl "https://api.mockaroo.com/api/8ba6f630?count=1000&key=11149690" > "products.csv"
! curl "https://api.mockaroo.com/api/6fa67fe0?count=3000&key=11149690" > "orders.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 99198    0 99198    0     0  69177      0 --:--:--  0:00:01 --:--:-- 69224
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  254k    0  254k    0     0   124k      0 --:--:--  0:00:02 --:--:--  124k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  229k    0  229k    0     0  37544      0 --:--:--  0:00:06 --:--:-- 52728


##Set up database

In [ ]:
import os
import pandas as pd
import sqlite3

In [ ]:
#see top 5 rows of customer file
pd.read_csv("customers.csv").head()

,customer_id,first_name,last_name,email,phone_number,address,city,country,postal_code,loyalty_points
0,1,Kirby,Christophle,kchristophle0@harvard.edu,100-375-9190,09 West Terrace,Flandes,Colombia,733517,497
1,2,Langston,Patshull,lpatshull1@woothemes.com,757-536-3066,00 Jay Court,Vitez,Bosnia and Herzegovina,NaN,768
2,3,Bettye,MacAloren,bmacaloren2@e-recht24.de,925-843-0316,670 Schlimgen Parkway,Sop Pong,Thailand,30270,91
3,4,Salli,Bolens,sbolens3@yahoo.com,999-383-0617,137 Fairfield Way,Huitang,China,NaN,147
4,5,Lowrance,Revill,lrevill4@ycombinator.com,177-652-1304,972 Paget Avenue,Bouças,Portugal,4950-270,780


##Creating the tables

**Customers Schema**

In [ ]:
customers_schema = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id INT PRIMARY KEY,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    email VARCHAR(50),
    phone_number VARCHAR(50),
    address VARCHAR(50),
    city VARCHAR(50),
    country VARCHAR(50),
    postal_code VARCHAR(50),
    loyalty_points INT
);
"""

#It is sql syntax

In [ ]:
pd.read_csv('products.csv').head()

,product_id,product_name,description,price,discount_percentage,category,brand,stock_quantity,color,size,weight,release_date,rating,reviews_count,seller_name,seller_rating,seller_reviews_count,shipping_method,shipping_cost
0,1,sapien non mi,Etiam faucibus cursus urna.,96.89,5.44,home decor,Yozio,183,Green,extra large,1.3,2014-09-08,2.9,343,eget tincidunt,4.2,40,standard,23.63
1,2,consectetuer adipiscing elit,Pellentesque ultrices mattis odio.,874.71,27.55,clothing,Kwilith,411,Fuscia,medium,3.0,2013-02-05,3.9,858,magnis dis parturient montes nascetur,4.2,727,express,4.76
2,3,nunc rhoncus dui,Sed vel enim sit amet nunc viverra dapibus. Nu...,369.90,0.27,clothing,Dynabox,147,Pink,small,9.9,2012-11-03,1.9,233,luctus,4.6,247,standard,14.78
3,4,suspendisse ornare,Sed vel enim sit amet nunc viverra dapibus. Nu...,916.90,18.58,clothing,Mita,131,Indigo,large,7.2,2013-05-27,4.1,795,ante ipsum,4.3,946,express,27.38
4,5,metus sapien,"Quisque erat eros, viverra eget, congue eget, ...",338.04,29.37,toys,Quire,852,Turquoise,extra large,2.3,2019-09-06,4.7,525,at,2.7,970,standard,17.35


**Products Schema**

In [ ]:
products_schema = """
CREATE TABLE IF NOT EXISTS products (
    product_id INT PRIMARY KEY,
    product_name TEXT,
    description TEXT,
    price DECIMAL(10,2),
    discount_percentage DECIMAL(5,2),
    category VARCHAR(50),
    brand TEXT,
    stock_quantity INT,
    color VARCHAR(50),
    size VARCHAR(20),
    weight DECIMAL(5,2),
    dimensions TEXT,
    release_date DATE,
    rating DECIMAL(3,1),
    reviews_count INT,
    seller_name TEXT,
    seller_rating DECIMAL(3,1),
    seller_reviews_count INT,
    shipping_method VARCHAR(20),
    shipping_cost DECIMAL(6,2)
);
"""

In [ ]:
pd.read_csv('orders.csv').head()

,order_id,customer_id,product_id,quantity,unit_price,total_price,order_date,shipping_address,payment_method,status
0,1,784,177,90,122.37,9895.60,2022-09-30,396 Welch Point,Apple Pay,pending
1,2,801,675,45,938.39,1247.98,2021-11-15,656 Derek Pass,Apple Pay,pending
2,3,107,327,47,227.71,3642.72,2020-10-31,5 Schmedeman Road,credit card,pending
3,4,705,29,51,732.56,7235.98,2021-03-06,825 Stang Plaza,Apple Pay,pending
4,5,335,124,28,924.83,3463.61,2022-09-29,2327 Buena Vista Circle,PayPal,delivered


**Orders Schema**

In [ ]:
orders_schema = """
CREATE TABLE IF NOT EXISTS orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    product_id INT,
    quantity INT,
    unit_price DECIMAL(10,2),
    total_price DECIMAL(10,2),
    order_date DATE,
    shipping_address VARCHAR(255),
    payment_method VARCHAR(20),
    status VARCHAR(20),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
"""

**Create the Database**

In [ ]:
db_name = 'ecommerce.db'

if os.path.exists(db_name):
  os.remove(db_name)
  print(f"Removed exixting database '{db_name}' .")

**or load the files as SQL formate**

In [ ]:
import os
import pandas as pd
import sqlite3


COLUMN_DATA_TYPES = {
    'customers': {
        'customer_id': 'int64',
        'first_name': 'object',
        'last_name': 'object',
        'email': 'object',
        'phone_number': 'object',
        'address': 'object',
        'city': 'object',
        'country': 'object',
        'postal_code': 'object',
        'loyalty_points': 'int64'
    },
    'products': {
        'product_id': 'int64',
        'product_name': 'object',
        'description': 'object',
        'price': 'float64',
        'discount_percentage': 'float64',
        'category': 'object',
        'brand': 'object',
        'stock_quantity': 'int64',
        'color': 'object',
        'size': 'object',
        'weight': 'float64',
        'dimensions': 'object',
        'release_date': 'datetime64[ns]',
        'rating': 'float64',
        'reviews_count': 'int64',
        'seller_name': 'object',
        'seller_rating': 'float64',
        'seller_reviews_count': 'int64',
        'shipping_method': 'object',
        'shipping_cost': 'float64'
    },
    'orders': {
        'order_id': 'int64',
        'customer_id': 'int64',
        'product_id': 'int64',
        'quantity': 'int64',
        'unit_price': 'float64',
        'total_price': 'float64',
        'order_date': 'datetime64[ns]',
        'shipping_address': 'object',
        'payment_method': 'object',
        'status': 'object'
    }
}


#database setup
db_name = 'ecommerce.db'
conn = None #Initialize the connection to None

try:
  #Established the connection b/w database and sqlite3
  conn = sqlite3.connect(db_name)
  #if exist the connected
  cursor = conn.cursor()
  print(f"The database '{db_name}' is created and connected successfully.")

  #now databse is connected the creating the table as like in sql workbench
  cursor.execute(customers_schema)
  cursor.execute(products_schema)
  cursor.execute(orders_schema)
  print("The table 'customer', 'products', 'order' created successfully. ")

  #--Load the data from CSV files into the tables using pandas
  csv_to_table_map = {
    '/content/customers.csv': 'customers', # Corrected filename
    '/content/products.csv': 'products',
    '/content/orders.csv': 'orders'     # Corrected filename
  }
  #files ka data 'customres, products, orders' table me store hoga

  for csv_file, table_name in csv_to_table_map.items():
    if os.path.exists(csv_file):
      print(f"\nProcessing '{csv_file}' into '{table_name}' ...")

      #Read the csv file into pandas dataframe
      df = pd.read_csv(csv_file)

      #use to_sql to insert clean dataframe

      df.to_sql(table_name, conn, if_exists='append', index=False)
      print(f" -> Data from '{csv_file}' loaded into '{table_name}' table successfully.")
    else:
      print(f"Warning '{csv_file}' not found. Skipping data load for '{table_name}'.")

  #commit the changes to the database
  conn.commit()
  print("\nData committed to database successfully.")


#Now handling the error
except sqlite3.Error as e:
  print(f"Database error: '{e}'.")
  if conn:
    conn.rollback()
    print("Database transaction rolled back due to error.")
except pd.errors.EmptyDataError as e:
  print(f"Pandas error: '{e}'. One of the CSV files might be empty.")
except KeyError as e:
  print(f"Schema definition error: A column is missing from the COLUMN_DATA_TYPES dictionary: '{e}'. ")
except Exception as e:
  print(f"An unexpected error occured: '{e}'.")
finally:
  #close the connection if it was established
  if conn:
    conn.close()
    print("Database connection closed.")

The database 'ecommerce.db' is created and connected successfully.
The table 'customer', 'products', 'order' created successfully. 

Processing '/content/customers.csv' into 'customers' ...
 -> Data from '/content/customers.csv' loaded into 'customers' table successfully.

Processing '/content/products.csv' into 'products' ...
 -> Data from '/content/products.csv' loaded into 'products' table successfully.

Processing '/content/orders.csv' into 'orders' ...
 -> Data from '/content/orders.csv' loaded into 'orders' table successfully.

Data committed to database successfully.
Database connection closed.


**Now write the Prompt**

In [ ]:
system_prompt = """
###ROLE###
You are a Senior Database Administrator and an expert SQL developer specializing in SQLite3. Your primary function is to act as a highly accurate Text-to-SQL translator, converting natural language questions into syntactically correct and highly optimized SQLite3 queries.

###CONTEXT###
You are querying a relational database for an e-commerce platform. The database uses the SQLite3 engine.

Here is the exact schema for the database:

```sql
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name TEXT,
    description TEXT,
    price DECIMAL(10,2),
    discount_percentage DECIMAL(5,2),
    category VARCHAR(50),
    brand TEXT,
    stock_quantity INT,
    color VARCHAR(50),
    size VARCHAR(20),
    weight DECIMAL(5,2),
    dimensions TEXT,
    release_date DATE,
    rating DECIMAL(3,1),
    reviews_count INT,
    seller_name TEXT,
    seller_rating DECIMAL(3,1),
    seller_reviews_count INT,
    shipping_method VARCHAR(20),
    shipping_cost DECIMAL(6,2)
);

CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    email VARCHAR(50),
    phone_number VARCHAR(50),
    address VARCHAR(50),
    city VARCHAR(50),
    country VARCHAR(50),
    postal_code VARCHAR(50),
    loyalty_points INT
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    product_id INT,
    quantity INT,
    unit_price DECIMAL(10,2),
    total_price DECIMAL(10,2),
    order_date DATE,
    shipping_address VARCHAR(255),
    payment_method VARCHAR(20),
    status VARCHAR(20),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

```

###TASK###

1. Carefully analyze the user's natural language request.
2. Identify the specific tables and columns required to fulfill the request based *only* on the provided schema.
3. Determine if any `JOIN` operations, `GROUP BY` clauses, or aggregate functions (`SUM`, `AVG`, `COUNT`) are necessary.
4. Construct a valid, efficient SQLite3 `SELECT` query that directly answers the user's question.

###CONSTRAINTS###

* **Read-Only Operations:** Strictly generate `SELECT` queries. Never generate `INSERT`, `UPDATE`, `DELETE`, `DROP`, or `ALTER` statements under any circumstances.
* **Strict Schema Adherence:** Do not hallucinate column names, table names, or relationships. Only use what is explicitly provided in the schema.
* **SQLite3 Dialect:** Ensure all functions and syntax are strictly compatible with SQLite3 (e.g., use `LIMIT` instead of `TOP`, use `strftime()` for complex date extraction).
* **Aliases:** Always use clear table aliases when performing `JOIN` operations (e.g., `FROM orders o JOIN customers c`).
* **Handling Ambiguity:** If a user query is too ambiguous to write a safe SQL statement, output a SQL comment detailing what clarification is needed instead of guessing.
* **Handling Unknowns:** If a user query is asking for something that does not exist in the schema, ask the user for clarifications. Do NOT make assumptions.

###EXAMPLES###

**User Input:** "Show me the top 5 most expensive products in the electronics category that are currently in stock."
**Output:** ```sql
-- Filtering by electronics category and stock > 0, ordering by price descending
SELECT product_name, price
FROM products
WHERE category = 'electronics' AND stock_quantity > 0
ORDER BY price DESC
LIMIT 5;

```

**User Input:** "What is the total revenue generated from customer John Doe?"
**Output:** ```sql
-- Joining orders and customers to sum the total price for a specific customer name
SELECT SUM(o.total_price) as total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE c.first_name = 'John' AND c.last_name = 'Doe';

```

**User Input:** "Which brands have an average product rating above 4.5?"
**Output:**

```sql
-- Grouping by brand and filtering on the average of the rating column
SELECT brand, AVG(rating) as avg_rating
FROM products
GROUP BY brand
HAVING AVG(rating) > 4.5;

```

###OUTPUT FORMAT###
Return only the sqllite SQL query as a code block using triple backticks and the `sql` language tag, like this:

```sql
-- Your SQL query here
```

"""

##Load the model with API Key

In [ ]:
llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash", api_key="AQ.Ab8RN6I02gwFaU81j5a9oCZ4q_2iJmWqmu1sZJzl6MYxNiqmKA")

In [ ]:
import json
from langchain_core.messages import HumanMessage

#define the function to generate the sql query by pass the query, system_prompt and model
def get_sql_query_via_langchain(llm, system_prompt, user_query):
  contents=f"""
  {system_prompt}

  Here's the user query in english you need to work on:

  {user_query}
  """

  #store the sql query generated by model
  response = llm.invoke(input=[HumanMessage(content = contents)])
  print(response)

  #access the usage_metadata attributes
  usage_metadata = response.usage_metadata

  #prints the differents token counts
  # print(f"Input toke counts: {usage_metadata.input_token_count}")
  # print(f"Thoughts toke counts: {response.usage_metadata.thoughts_token_count}")
  # print(f"Output toke counts: {usage_metadata.candidates_token_count}")
  # print(f"Total toke counts: {usage_metadata.total_token_count}")

  output = response.text.replace('```sql', '').replace('```', '')

  return output

In [ ]:
import sqlite3
import pandas as pd

def execute_query(query, db_name='ecommerce.db'):

    conn = None
    try:
        # Connect to the database
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()

        # Execute the query
        print(f"\nExecuting query on '{db_name}':\n{query}")
        cursor.execute(query)

        # Fetch all results
        results = cursor.fetchall()

        # Get column names from the cursor description
        columns = [description[0] for description in cursor.description]

        # Format results as a dataframe for easier use
        results_as_dict = [dict(zip(columns, row)) for row in results]
        results_df = pd.DataFrame(results_as_dict)

        print("Query executed successfully.")
        return results_df

    except sqlite3.Error as e:
        print(f"Database error executing query: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    finally:
        if conn:
            conn.close()

In [ ]:
def text2sql(llm, system_prompt, user_query):
  sqllite_query = get_sql_query_via_langchain(llm, system_prompt, user_query)
  results = execute_query(sqllite_query)
  return results

In [ ]:
text2sql(llm, system_prompt, "Show me the order count by country")

content=[{'type': 'text', 'text': '```sql\n-- Joining orders and customers to count the number of orders per country\nSELECT c.country, COUNT(o.order_id) AS order_count\nFROM orders o\nJOIN customers c ON o.customer_id = c.customer_id\nGROUP BY c.country\nORDER BY order_count DESC;\n```', 'extras': {'signature': 'Eo4NCosNARFNMg8lx7DynT3UAfhH210VQfWErLGfqe8iYbEL1cF82F1IVblDqd3E+aKeFdMQamMMuUl9dJFZgXGyVmMCwoELj7Tq2NhIcvHfczgbJaGptqEHzm/hO8DxSI3Q22PqgukF0FQGE+CEDqtbL6p/bQXUVf5dn/4+b7TYRqsZGAUn5doeogcx97hLfVjfyOB+tNejKlmYSNlZzMvxeqqs/RP7qxdED7y+4N2VTn10kr9e345Nwrsj4Q/hVRTs0phKpgqGtUD9QfkbMddRmycjMrg0vCeYhDAmB2UhP16LQpUYJrykqGZysaWa9mxvaZHIvCum+n+YdlWySNnCjhXr8C82Wf5hkrxIvBpGcV0mPxSFwbsuKAL0zgUOPx3F5BpGkR1X2T4nS1d5q3LCyy4zxCp9WheamWu6ukDDOifKYwsznLqeVtgFjYpY81YxFarM15C5K8f92DzaQOnMnZBqxFXxjLxLMEGqWXYOwlq/sadoiL+rxPq8teI8GypSQtdW8m90UxH5QJKG7XebIoAlquAfIm8Bbgsc6WQZbOYk7loHvchhmbWRDkO9nCLdt2JjQr+QFE/RzaDFtwaQVu9PGHoCVX5v5crSGVB+N1iwW3XsSX0W2xiSarzKLi9AiTCWfRuFJzUHjmlnA0kf4tgQOdGj6mdfB5Ibx0rSR

,country,order_count
0,China,552
1,Indonesia,328
2,Russia,198
3,Philippines,175
4,Sweden,110
...,...,...
115,Zambia,1
116,Swaziland,1
117,Lebanon,1
118,Italy,1


In [ ]:
text2sql(llm, system_prompt, "List orders from countries that are in Asia")

content=[{'type': 'text', 'text': '```sql\n-- Clarification needed: The database schema does not contain a "continent" column or a mapping of countries to continents. \n-- Please specify the list of Asian countries to filter by (e.g., \'Japan\', \'India\', \'China\', etc.) or define how to identify Asian countries from the available data.\n```', 'extras': {'signature': 'EoYdCoMdARFNMg83dE2ad3Vw6TemqMjT4Ag6KBuIxEkrARTWKoiJaLFMouFFe93OSYYhFC+Y4b31v/MzstPBo5hgWHNwXb+arMdkGydI45jYr2bg3dCd9R7eoSrNLmeq7d1wiNgjp+2IjIW78WHqO84KGGipygJfeF5Xs3dUJrK2U/OPU2Mk6lqQZNiDhWAPdypdNNPu2kfjP0467CU9OjPYzu44sBke3wm2Odryumw/GJ9iIs3A0sXiPpVYOmyRhTzUpZFAEW6Ltt/RGUtRq1Mot34OIWdMY7xEtjvpDuShCWhuJppWQ9fwWF9s8aRJPcdmVEd0wYUBdRQqEPNoc8fefv8zGHjuBpfaSQsnvy0j2kLlG3Q4S/z/AoCBPBNS5jiBYAOZgVyzXv9qGz7xejagMSFixKqAnj2lUlA0iYBiXNE4MUIykUh/NZXZFb5/kQ7zvfn3WUBfMiA+vX48fKSRlpeTFFrFtXLCbVX+za0PWCItWNItkMbgtnf90mqT7g+ULINuRehx2xMSiHSAF01FIjXVCW9OlGlm7durc0GWQM48Ycwq1gBaAS9mEdSDVPhMLNIWR3A6ljjm7DTs7SCz3Xnrr5GdBOB8pKER9E4rcFAP3lM

In [ ]:
text2sql(llm, system_prompt, "List orders from countries that are in Asia")

content=[{'type': 'text', 'text': "```sql\n-- Clarification Needed: The database schema contains a 'country' column in the 'customers' table, but it does not contain continent information to identify which countries are in 'Asia'. Please provide a list of specific Asian countries to filter by, or define how countries are mapped to continents in your dataset.\n```", 'extras': {'signature': 'EtgjCtUjARFNMg8xgdbZkgaHxX50V2KIoyQugYlDhpYAex0LJqYNo/TczmnLcCUGt+/iyqLtmj26h3i6K3lEUHXZ6q+pXCLI/W4DTcQ0sUtgGIQ68nZ+VmciGPw6g9vyZquC8BbzJXREr8+em5FvMxmIIt+ONW+Jil+BIpS22rugDwouC3MO+I1dhSlvHQpgi3ZleKgBrqB6effNfZ1879uukEMrQ0AeTs2wNnc7x5AnkZwNcn+2Vt6MBshxOvsBJ+Mrj7yLiDf1YEizRv85PdFBv8t5Oi9VQTEQWCUbemvRHumQ7PjTHyxzyEdwfdCfN8eXQ/qDG20f+G8dWuNz1xwaPvlI2NME54D/2KWE42kfgD3qDiDu0yIFDt1SlWHEpPgrRQaoilhgGGSsoB4hmCftwxZ5+DyV0VA39o4o86pbFEaMZXJdGSr9xQ8PNIecDWGDupr4AZiXB1O0NgJbcPWsMfXsHp7g8/qn6PbWV/Zj6NGDHCt9/BjH/+LcxBLhEk3l22Hox2ShOrY7nIlBdExaVVrQV0xHuwn++6r0m9e/3whrTjm8IsCTWs/F/RfdPaRtA+BWOZr5d1CId7OdX6HwGCX4tPI

In [ ]:
text2sql(llm, system_prompt, "What are my top 10 most popular products")

content=[{'type': 'text', 'text': '```sql\n-- Top 10 most popular products based on the total quantity sold\nSELECT p.product_id, p.product_name, SUM(o.quantity) AS total_quantity_sold\nFROM products p\nJOIN orders o ON p.product_id = o.product_id\nGROUP BY p.product_id, p.product_name\nORDER BY total_quantity_sold DESC\nLIMIT 10;\n```', 'extras': {'signature': 'Eq8QCqwQARFNMg9x56RXhbp7RpE2hxgs7XY12kZ6e1k5/PnBPAme0bNCuWa6xc+XT3oEMcRAAQgOKfcIlpfuxoGXPXEXoSsRRquckg0BdV+GPFj+iIACnBczRevDqxq58pziUEWr4BAUleRWZQsY5P60Kd5sPcNuGaCgM5wIJe3evt6EHxVMqVJ+2wmS2/ojinGuiEo6EVHGAJki5t73dEA1JRxeVY2+BvjmIH6kK7hS7HKBBfuPaq5ydKgHkBFdsnNT+zWIHsM6hCzLi4NZ8wI7jfQQo/jLJk19OlnDZqUYhvMWy/Mnn43dit0Fr5QW+RRY/i34OcGc09MZJTVgCO3GvRKVbSQiU6WHgtfZTuoKDIpjSCTEqwC0Y5XFQ5/unUg2OiN4FUAh+0oWLLvq/lLA5nBG25vDrDsafnysszi2TexrGmXyFZ1mP9b0ssBO0Rq5q3k9KOa5Bf39x9klTw3FN/Ju1sKGALm9PQWJTC3LrCpSHfyo3Hg7SMMqsvAoTy71CYLibXuVKyOTP3Wmrs+o9VNu6KMcHL82paWAjBHFYC9KXiYRm1Lb6VeZ9dfp8fJz205+ulhMCm2+DNkDxntdDHXokGySLcBC10DqBsxso1luoYBHm5jmqeJ

,product_id,product_name,total_quantity_sold
0,880,euismod scelerisque,439
1,1,sapien non mi,433
2,924,felis eu sapien,428
3,185,aliquet at feugiat,421
4,539,porttitor id consequat,420
5,111,dapibus augue vel accumsan,418
6,227,dapibus dolor,398
7,632,aliquam sit,394
8,397,dui proin leo odio,390
9,651,diam vitae quam,388
